In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import r3d_18, R3D_18_Weights
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc as calc_auc
from sklearn.preprocessing import label_binarize
from sklearn.metrics import ConfusionMatrixDisplay
from transformers import AutoTokenizer, AutoModel
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ==========================================
# 0. MOUNT GOOGLE DRIVE & UNZIP NIFTI FOLDER
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

# ── Update this path if your zip is inside a subfolder in Drive ──
# e.g. '/content/drive/MyDrive/my_folder/nifti_subjects.zip'
ZIP_PATH     = '/content/drive/MyDrive/nifti_subjects_post_1.zip'
EXTRACT_PATH = '/content/'

print("Unzipping NIfTI folder... (may take 5-10 mins for 21GB)")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)
print("Done unzipping!\n")

In [ ]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
EXCEL_PATH   = '/content/drive/MyDrive/Clinical_and_Other_Features.xlsx'
NIFTI_FOLDER = '/content/nifti_subjects_post_1'
SEQ_TYPE     = 'post_1'
TEXT_CSV     = '/content/drive/MyDrive/mistral_patient_radiology_reports.csv'

ID_COL    = 'Patient ID'
LABEL_COL = 'Mol Subtype'

BATCH_SIZE    = 2          # Keep small for 3D volumes
EPOCHS        = 50         # Was 2 — needs many more epochs
LEARNING_RATE_IMAGE = 1e-4  # Image encoder LR
LEARNING_RATE_TEXT  = 2e-5  # Text encoder LR — lower, already pretrained
LEARNING_RATE_HEAD  = 5e-4  # Fusion head LR
TARGET_SHAPE  = (32, 128, 128)
MAX_TEXT_LEN  = 256
PATIENCE      = 8          # Early stopping patience
GRAD_CLIP     = 1.0        # Gradient clipping max norm

# *** KEY CHANGE: Domain-matched text encoder ***
# BiomedBERT pretrained on PubMed abstracts + PMC full text
# Much better than DistilBERT for clinical/radiology language
TEXT_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
# Alternative: "emilyalsentzer/Bio_ClinicalBERT" (MIMIC-trained, even more clinical)
# Alternative: "allenai/scibert_scivocab_uncased" (scientific text)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Text encoder: {TEXT_MODEL_NAME}")

In [ ]:
# ==========================================
# 2. SCAN NIFTI FOLDER
# ==========================================
print("Scanning NIfTI folder...")
available_ids = []
for subject in sorted(os.listdir(NIFTI_FOLDER)):
    subject_dir = os.path.join(NIFTI_FOLDER, subject)
    if not os.path.isdir(subject_dir):
        continue
    nii_path = os.path.join(subject_dir, SEQ_TYPE, f"{subject}_{SEQ_TYPE}.nii.gz")
    if os.path.isfile(nii_path):
        available_ids.append(subject)

available_ids = set(available_ids[:-2])  # Drop last 2 corrupted
print(f"NIfTI files found: {len(available_ids)} subjects")

In [ ]:
# ==========================================
# 3. LOAD TOKENIZER & TEXT DATA
# ==========================================
print(f"Loading tokenizer: {TEXT_MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

text_df = pd.read_csv(TEXT_CSV)
text_df['patient_id'] = text_df['patient_id'].astype(str)
text_dict = dict(zip(text_df['patient_id'], text_df['generated_report']))
print(f"Loaded {len(text_dict)} radiology reports")

In [ ]:
# ==========================================
# 4. DATASET
# ==========================================
class BreastMRIDataset(Dataset):
    def __init__(self, df, img_dir, seq_type, target_shape, text_dict, tokenizer, max_len=256):
        self.df           = df.reset_index(drop=True)
        self.img_dir      = img_dir
        self.seq_type     = seq_type
        self.target_shape = target_shape
        self.text_dict    = text_dict
        self.tokenizer    = tokenizer
        self.max_len      = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row        = self.df.loc[idx]
        patient_id = row[ID_COL]
        label      = row['encoded_label']

        # --- Image ---
        file_name = f"{patient_id}_{self.seq_type}.nii.gz"
        img_path  = os.path.join(self.img_dir, patient_id, self.seq_type, file_name)
        try:
            img = nib.load(img_path).get_fdata(dtype=np.float32)
        except Exception:
            img = np.zeros(self.target_shape, dtype=np.float32)

        # Normalize to [0, 1]
        vmin, vmax = img.min(), img.max()
        img = (img - vmin) / (vmax - vmin + 1e-8)

        img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
        img_resized = F.interpolate(
            img_tensor, size=self.target_shape, mode='trilinear', align_corners=False
        ).squeeze(0)  # -> (1, D, H, W)

        # --- Text ---
        # Ensure patient_id is a plain str (guards against np.int64 / float NaN)
        pid_str = str(patient_id).strip()
        text = self.text_dict.get(pid_str, "")
        # Final safety net: tokenizer requires a str, never None / float
        if not isinstance(text, str) or not text:
            text = "No report available."
        tokens = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        return (
            img_resized,
            tokens['input_ids'].squeeze(0),
            tokens['attention_mask'].squeeze(0),
            torch.tensor(label, dtype=torch.long)
        )

In [ ]:
# ==========================================
# 5. FOCAL LOSS WITH LABEL SMOOTHING
# ==========================================
class FocalLoss(nn.Module):
    """
    Focal loss with optional label smoothing.
    Label smoothing helps prevent the model from being overconfident
    on majority classes, improving generalization on imbalanced data.
    """
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.1, reduction='mean'):
        super().__init__()
        self.weight          = weight
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
        self.reduction       = reduction

    def forward(self, inputs, targets):
        # Label smoothing: soft targets
        num_classes = inputs.size(1)
        smooth_val  = self.label_smoothing / (num_classes - 1)
        soft_targets = torch.full_like(inputs, smooth_val)
        soft_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)

        # Standard CE with class weights for focal scaling
        ce_loss    = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt         = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean': return focal_loss.mean()
        elif self.reduction == 'sum': return focal_loss.sum()
        return focal_loss

In [ ]:
# ==========================================
# 6. SETUP & SPLITS
# KEY CHANGE: 70/15/15 instead of 40/20/40
# This gives ~75% more training data
# ==========================================
print("Loading clinical data...")
df = pd.read_excel(EXCEL_PATH, header=1)
df[ID_COL] = df[ID_COL].astype(str)
df = df.dropna(subset=[LABEL_COL, ID_COL])
df = df[df[ID_COL].isin(available_ids)].copy().reset_index(drop=True)
print(f"Total usable subjects: {len(df)}")

le = LabelEncoder()
df['encoded_label'] = le.fit_transform(df[LABEL_COL])
num_classes = len(le.classes_)
print(f"Classes ({num_classes}): {list(le.classes_)}")

# Class distribution — useful to see imbalance
print("\nClass distribution:")
print(df[LABEL_COL].value_counts())

# 70 / 15 / 15 split
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['encoded_label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['encoded_label'], random_state=42
)
print(f"\nSplits -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Class weights on training set
classes       = np.unique(train_df['encoded_label'])
weights       = compute_class_weight('balanced', classes=classes, y=train_df['encoded_label'])
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print(f"Class weights: {dict(zip(le.classes_, weights.round(3)))}")

In [ ]:
# Dataloaders
train_dataset = BreastMRIDataset(train_df, NIFTI_FOLDER, SEQ_TYPE, TARGET_SHAPE, text_dict, tokenizer, MAX_TEXT_LEN)
val_dataset   = BreastMRIDataset(val_df,   NIFTI_FOLDER, SEQ_TYPE, TARGET_SHAPE, text_dict, tokenizer, MAX_TEXT_LEN)
test_dataset  = BreastMRIDataset(test_df,  NIFTI_FOLDER, SEQ_TYPE, TARGET_SHAPE, text_dict, tokenizer, MAX_TEXT_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
# ── Preview one subject: MRI slices + radiology report ────────────────────────
PREVIEW_IDX = 1  # change to any index in test_df

row        = test_df.iloc[PREVIEW_IDX]
patient_id = row[ID_COL]
true_label = le.classes_[row['encoded_label']]

# ── Load & resize volume ──────────────────────────────────────────────────────
file_name  = f"{patient_id}_{SEQ_TYPE}.nii.gz"
img_path   = os.path.join(NIFTI_FOLDER, patient_id, SEQ_TYPE, file_name)
img        = nib.load(img_path).get_fdata(dtype=np.float32)
img        = (img - img.min()) / (img.max() - img.min() + 1e-8)
img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
img_resize = F.interpolate(img_tensor, size=TARGET_SHAPE,
                            mode='trilinear', align_corners=False).squeeze().numpy()

# ── Radiology report ──────────────────────────────────────────────────────────
report = text_dict.get(str(patient_id).strip(), "No report found.")

# ── Content-aware centre slices ───────────────────────────────────────────────
d = int(np.argmax(img_resize.sum(axis=(1, 2))))
h = int(np.argmax(img_resize.sum(axis=(0, 2))))
w = int(np.argmax(img_resize.sum(axis=(0, 1))))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (title, sl) in zip(axes, [
    ('Axial',    img_resize[d, :, :]),
    ('Coronal',  img_resize[:, h, :]),
    ('Sagittal', img_resize[:, :, w]),
]):
    ax.imshow(sl.T, cmap='gray', origin='lower')
    ax.set_title(title, fontsize=11)
    ax.axis('off')

fig.suptitle(f'{patient_id}  |  Label: {true_label}', fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nPatient  : {patient_id}")
print(f"Label    : {true_label}")
print(f"\nRadiology Report:\n{'─'*60}\n{report}\n{'─'*60}")

In [ ]:
# ==========================================
# 7. MODEL ARCHITECTURE
# KEY CHANGES:
#   - BiomedBERT text encoder (domain-matched)
#   - Mean pooling over all tokens (vs CLS-only)
#   - Fusion MLP with 2 hidden layers + residual skip
#   - Dropout for regularization
# ==========================================

def mean_pool(last_hidden_state, attention_mask):
    """
    Mean pooling over non-padding tokens.
    Better than CLS-only for short clinical sentences where
    information is distributed across all tokens.
    """
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    count  = mask.sum(dim=1).clamp(min=1e-9)
    return summed / count

class FusionMLP(nn.Module):
    """
    2-layer MLP fusion head with residual skip connection.
    Much richer than a single linear layer.
    """
    def __init__(self, img_dim, text_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        in_dim = img_dim + text_dim

        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Residual projection to match hidden_dim if in_dim != hidden_dim
        self.skip = nn.Linear(in_dim, hidden_dim, bias=False)
        self.head = nn.Linear(hidden_dim, num_classes)

    def forward(self, img_feat, text_feat):
        x = torch.cat([img_feat, text_feat], dim=1)
        out = self.proj(x) + self.skip(x)  # residual
        return self.head(out)

class MultiModalModel(nn.Module):
    def __init__(self, num_classes, text_model_name=TEXT_MODEL_NAME):
        super().__init__()

        # --- Image encoder: 3D ResNet-18 ---
        base = r3d_18(weights=R3D_18_Weights.DEFAULT)
        #Adapt from 3-channel video to 1-channel MRI
        original_conv = base.stem[0]
        base.stem[0] = nn.Conv3d(
            1, original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=False
        )
        base.stem[0].weight.data = original_conv.weight.data[:, :1, :, :, :]
        img_dim = base.fc.in_features  # 512
        base.fc = nn.Identity()
        self.img_encoder = base

        # --- Image encoder: DenseNet-121 ---
        #self.img_encoder = DenseNet121(
        #  spatial_dims=3,       # 3D volumes
        #  in_channels=1,        # grayscale MRI
        #  out_channels=512,     # embedding size fed into fusion MLP
        #)
        #self.img_encoder.class_layers = nn.Identity()  # strip classification head
        #img_dim = 1024

        # --- Text encoder: BiomedBERT ---
        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        text_dim = self.text_encoder.config.hidden_size  # 768

        # --- Fusion MLP ---
        self.fusion = FusionMLP(
            img_dim=img_dim,
            text_dim=text_dim,
            hidden_dim=512,
            num_classes=num_classes,
            dropout=0.3
        )

    def forward(self, x, input_ids, attention_mask):
        img_feat  = self.img_encoder(x)  # (B, 512)
        if img_feat.dim() == 5:          # DenseNet returns (B, C, D, H, W)
          img_feat = img_feat.mean(dim=[2, 3, 4])

        text_out  = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_feat = mean_pool(text_out.last_hidden_state, attention_mask)  # (B, 768)

        return self.fusion(img_feat, text_feat)


model = MultiModalModel(num_classes=num_classes).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# ==========================================
# 8. TRAINING SETUP
# KEY CHANGES:
#   - Differential learning rates (image, text, head)
#   - CosineAnnealingLR scheduler
#   - Gradient clipping
# ==========================================

criterion = FocalLoss(weight=class_weights, gamma=2.0, label_smoothing=0.1)

# Differential LRs: text encoder needs a much smaller LR
# because it's already well-pretrained
optimizer = optim.AdamW([
    {'params': model.img_encoder.parameters(),  'lr': LEARNING_RATE_IMAGE},
    {'params': model.text_encoder.parameters(), 'lr': LEARNING_RATE_TEXT},
    {'params': model.fusion.parameters(),       'lr': LEARNING_RATE_HEAD},
], weight_decay=1e-4)

# Cosine annealing: gradually reduces LR from max to near-zero
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

scaler = torch.amp.GradScaler('cuda')

In [ ]:
# ==========================================
# 9. TRAINING LOOP WITH EARLY STOPPING
# ==========================================
print("\nStarting Training...")

history = {'train_loss': [], 'val_acc': [], 'val_auc': []}
best_val_auc   = 0.0
patience_count = 0
best_model_state = None

for epoch in range(EPOCHS):
    # ---- TRAIN ----
    model.train()
    running_loss = 0.0

    for inputs, input_ids, attention_mask, labels in train_loader:
        inputs        = inputs.to(device, non_blocking=True)
        input_ids     = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        labels        = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(inputs, input_ids, attention_mask)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()

        # Gradient clipping — prevents exploding gradients
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    scheduler.step()

    # ---- VALIDATE ----
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for inputs, input_ids, attention_mask, labels in val_loader:
            inputs        = inputs.to(device, non_blocking=True)
            input_ids     = input_ids.to(device, non_blocking=True)
            attention_mask = attention_mask.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                outputs = model(inputs, input_ids, attention_mask)
                probs   = F.softmax(outputs, dim=1)

            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    try:
        if num_classes > 2:
            auc_score = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
        else:
            auc_score = roc_auc_score(all_labels, np.array(all_probs)[:, 1])
    except ValueError:
        auc_score = 0.0

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] Loss: {epoch_loss:.4f} | "
          f"Val Acc: {acc:.4f} | Val AUC: {auc_score:.4f} | LR: {current_lr:.2e}")

    history['train_loss'].append(epoch_loss)
    history['val_acc'].append(acc)
    history['val_auc'].append(auc_score)

    # --- Early stopping on val AUC ---
    if auc_score > best_val_auc:
        best_val_auc = auc_score
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_count = 0
        print(f"  ✓ New best AUC: {best_val_auc:.4f} — saving checkpoint")
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
            break

# Restore best weights
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nRestored best model (Val AUC: {best_val_auc:.4f})")

In [ ]:
# ==========================================
# 10. LEARNING CURVES
# ==========================================
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(history['train_loss'])+1), history['train_loss'],
         marker='o', color='red', label='Train Loss')
plt.title('Training Loss over Epochs', fontsize=14)
plt.xlabel('Epoch'); plt.ylabel('Focal Loss'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, len(history['val_acc'])+1), history['val_acc'],
         marker='s', color='blue',  label='Val Accuracy')
plt.plot(range(1, len(history['val_auc'])+1), history['val_auc'],
         marker='^', color='green', label='Val Macro AUC')
plt.axhline(best_val_auc, color='green', linestyle='--', alpha=0.5, label=f'Best AUC={best_val_auc:.3f}')
plt.title('Validation Metrics over Epochs', fontsize=14)
plt.xlabel('Epoch'); plt.ylabel('Score'); plt.legend()

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150)
plt.show()

In [ ]:
# ==========================================
# 11. CONFUSION MATRIX (Validation)
# ==========================================
cm = confusion_matrix(all_labels, all_preds, labels=range(num_classes))
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)

fig, ax = plt.subplots(figsize=(8, 6))
cm_display.plot(ax=ax, cmap='Blues', colorbar=True)
ax.grid(False)
plt.title(f'Validation Confusion Matrix\nAcc: {history["val_acc"][-1]:.4f} | AUC: {best_val_auc:.4f}',
          fontsize=14, pad=15)
plt.savefig('val_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ==========================================
# 12. FINAL TEST EVALUATION
# ==========================================
print("\n" + "="*50)
print("EVALUATION ON UNSEEN TEST DATA")
print("="*50)

model.eval()
test_preds, test_labels, test_probs = [], [], []

with torch.no_grad():
    for inputs, input_ids, attention_mask, labels in test_loader:
        inputs         = inputs.to(device)
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        with torch.amp.autocast('cuda'):
            outputs = model(inputs, input_ids, attention_mask)
            probs   = F.softmax(outputs, dim=1)

        _, preds = torch.max(outputs, 1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.numpy())
        test_probs.extend(probs.cpu().numpy())

test_acc = accuracy_score(test_labels, test_preds)
try:
    if num_classes > 2:
        test_auc = roc_auc_score(test_labels, test_probs, multi_class='ovr', average='macro')
    else:
        test_auc = roc_auc_score(test_labels, np.array(test_probs)[:, 1])
except ValueError:
    test_auc = 0.0

print(f"FINAL TEST ACCURACY : {test_acc:.4f}")
print(f"FINAL TEST MACRO AUC: {test_auc:.4f}\n")

test_cm    = confusion_matrix(test_labels, test_preds, labels=range(num_classes))
cm_display = ConfusionMatrixDisplay(confusion_matrix=test_cm, display_labels=le.classes_)

fig, ax = plt.subplots(figsize=(8, 6))
cm_display.plot(ax=ax, cmap='Greens', colorbar=True)
ax.grid(False)
plt.title(f'Test Set Confusion Matrix\nAccuracy: {test_acc:.4f} | Macro AUC: {test_auc:.4f}',
          fontsize=14, pad=15)
plt.savefig('test_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ==========================================
# 14. SAVE BEST MODEL
# ==========================================
SAVE_PATH = '/content/drive/MyDrive/best_multimodal_model_BiomedBERT.pt'

torch.save({
    'model_state_dict': best_model_state,
    'label_encoder_classes': le.classes_,
    'num_classes': num_classes,
    'val_auc': best_val_auc,
    'text_model_name': TEXT_MODEL_NAME,
}, SAVE_PATH)

print(f"Model saved to {SAVE_PATH}")

# To reload later:
# checkpoint = torch.load(SAVE_PATH)
# model = MultiModalModel(num_classes=checkpoint['num_classes'])
# model.load_state_dict(checkpoint['model_state_dict'])
# model.eval()

In [ ]:


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_PATH = './best_multimodal_model_BiomedBERT.pt'
checkpoint = torch.load(SAVE_PATH, map_location=device, weights_only=False)
model = MultiModalModel(num_classes=checkpoint['num_classes']).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded model (Val AUC: {checkpoint['val_auc']:.4f})")

In [ ]:
# ==========================================
# 13. GRAD-CAM  –  Multimodal (r3d_18 + Mistral)
#     Same FIXED_INDICES as baseline for cross-model comparison
# ==========================================
import matplotlib.cm       as mpl_cm
import matplotlib.gridspec as gridspec

FIXED_INDICES = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

print("Fixed GradCAM patients (multimodal resnet+llama):")
for i in FIXED_INDICES:
    pid   = test_df.iloc[i][ID_COL]
    label = le.classes_[test_df.iloc[i]['encoded_label']]
    print(f"  [{i:2d}]  {pid}  |  True label: {label}")


# ── GradCAM class ─────────────────────────────────────────────────────────────
class GradCAM3D_Multimodal:
    def __init__(self, model, target_layer):
        self.model       = model
        self.gradients   = None
        self.activations = None
        self._hooks = [
            target_layer.register_forward_hook(self._fwd_hook),
            target_layer.register_full_backward_hook(self._bwd_hook),
        ]

    def _fwd_hook(self, module, inp, out):
        self.activations = out.detach()

    def _bwd_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, img_tensor, input_ids, attention_mask, target_class=None):
        self.model.eval()
        img  = img_tensor.to(device).requires_grad_(True)
        ids  = input_ids.to(device)
        mask = attention_mask.to(device)

        output = self.model(img, ids, mask)

        if target_class is None:
            target_class = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, target_class].backward()

        weights = self.gradients[0].mean(dim=(1, 2, 3))
        cam     = (weights[:, None, None, None] * self.activations[0]).sum(dim=0)
        cam     = F.relu(cam)

        lo, hi = cam.min(), cam.max()
        if hi - lo > 1e-8:
            cam = (cam - lo) / (hi - lo)

        probs = output.softmax(dim=1)[0].detach().cpu().numpy()
        return cam.cpu().numpy(), target_class, probs

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()


# ── Overlay helper ────────────────────────────────────────────────────────────
def blend_mm(mri_sl, cam_sl, alpha=0.25):
    rgb  = np.stack([mri_sl] * 3, axis=-1)
    heat = mpl_cm.jet(cam_sl)[..., :3]
    return np.clip((1 - alpha) * rgb + alpha * heat, 0, 1)


# ── Hook model.base.layer4  (r3d_18 is stored as self.base in MultiModalModel)
gradcam  = GradCAM3D_Multimodal(model, model.img_encoder.layer4)

SAVE_DIR = './gradcam_multimodal_resnet_mistral_biomedBert'
os.makedirs(SAVE_DIR, exist_ok=True)

summary_rows = []

for idx in FIXED_INDICES:
    patient_id = test_df.iloc[idx][ID_COL]
    true_label = test_df.iloc[idx]['encoded_label']

    # ── Load image ────────────────────────────────────────────────────────────
    file_name  = f"{patient_id}_{SEQ_TYPE}.nii.gz"
    img_path   = os.path.join(NIFTI_FOLDER, patient_id, SEQ_TYPE, file_name)
    try:
        img = nib.load(img_path).get_fdata(dtype=np.float32)
    except Exception:
        img = np.zeros(TARGET_SHAPE, dtype=np.float32)

    img        = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
    img_resized = F.interpolate(img_tensor, size=TARGET_SHAPE,
                                mode='trilinear', align_corners=False)

    # ── Load + tokenise text ──────────────────────────────────────────────────
    pid_str = str(patient_id).strip()
    text    = text_dict.get(pid_str, "No report available.")
    if not isinstance(text, str) or not text:
        text = "No report available."

    tokens         = tokenizer(text, padding='max_length', truncation=True,
                               max_length=256, return_tensors='pt')
    input_ids      = tokens['input_ids']
    attention_mask = tokens['attention_mask']

    # ── Generate CAM ──────────────────────────────────────────────────────────
    cam_3d, pred_class, probs = gradcam.generate(img_resized, input_ids, attention_mask)

    cam_up = F.interpolate(
        torch.tensor(cam_3d).unsqueeze(0).unsqueeze(0),
        size=TARGET_SHAPE, mode='trilinear', align_corners=False
    ).squeeze().numpy()

    mri       = img_resized.detach().squeeze().numpy()
    true_name = le.classes_[true_label]
    pred_name = le.classes_[pred_class]
    correct   = '✓' if true_label == pred_class else '✗'
    conf      = float(probs[pred_class])

    summary_rows.append({
        'Index'     : idx,
        'Patient'   : patient_id,
        'True'      : true_name,
        'Pred'      : pred_name,
        'Correct'   : correct,
        'Confidence': f'{conf:.3f}',
    })

    # ── Content-aware centre slices ───────────────────────────────────────────
    d = int(np.argmax(mri.sum(axis=(1, 2))))
    h = int(np.argmax(mri.sum(axis=(0, 2))))
    w = int(np.argmax(mri.sum(axis=(0, 1))))

    views = {
        'Axial'   : (mri[d, :, :],  cam_up[d, :, :]),
        'Coronal' : (mri[:, h, :],  cam_up[:, h, :]),
        'Sagittal': (mri[:, :, w],  cam_up[:, :, w]),
    }

    # ── Save each view as its own file (same as baseline) ────────────────────
    for view_name, (mri_sl, cam_sl) in views.items():
        H, W = mri_sl.T.shape
        MIN_FIG_W = 6.0
        MIN_FIG_H = 4.0

        scale = 4.0 / max(H, W)
        fig_h = max(H * scale, MIN_FIG_H)
        fig_w = max(W * scale * 2 + 0.1, MIN_FIG_W)

        fig, axes = plt.subplots(1, 2, figsize=(fig_w, fig_h),
                                 gridspec_kw={'wspace': 0.04})

        axes[0].imshow(mri_sl.T, cmap='gray', origin='lower', aspect='auto')
        axes[0].set_title(view_name, fontsize=9)
        axes[0].axis('off')

        axes[1].imshow(blend_mm(mri_sl.T, cam_sl.T), origin='lower', aspect='auto')
        axes[1].set_title('GradCAM', fontsize=9)
        axes[1].axis('off')

        fig.suptitle(
            f'[{idx}] {patient_id}  |  True: {true_name}  |  Pred: {pred_name} {correct}\n'
            f'Probs: { {k: round(float(v), 3) for k, v in zip(le.classes_, probs)} }',
            fontsize=9, y=1.04
        )

        plt.tight_layout()
        fname = f'gradcam_mm_idx{idx:02d}_{patient_id}_{view_name.lower()}.png'
        plt.savefig(os.path.join(SAVE_DIR, fname), dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f'  Saved → {fname}')

gradcam.remove_hooks()

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("GRAD-CAM SUMMARY  –  Multimodal (r3d_18 + Mistral)")
print("="*65)
print(pd.DataFrame(summary_rows).to_string(index=False))
print("="*65)
print(f"\nAll figures saved to: {SAVE_DIR}/")